# Phase 1: Qwen3.5-2B operator agent (v5 data, length-grouped batches)

Fine-tunes the operator agent on the Way 1 dataset and exports a GGUF for llama.cpp.

- Data: `domains/operator_agent/data/way1_train_v5.jsonl` (op/1 examples, 245 operators incl. bai.ask / bai.decline / bai.loopcut)
- Model: `unsloth/Qwen3.5-2B`, LoRA r=32 on the text layers (vision layers stay frozen)
- Output: merged `q8_0` GGUF (~2 GB) for `llama-server`

**Runs on free GPUs, about 5.5-6 hours on a T4** (v4 took 4.5 h; v5 has ~1.5x the real tokens).

Why this notebook exists: `operator_finetune.ipynb` ran v5 at ~21 s/step (~14 h). v5 mixes short
requests (median ~320 tokens) with big scenes (up to ~1,140), and random batches pad every example
to the longest in its batch: ~16.7M tokens per epoch processed for 8.4M real ones (v4: 6.2M). The
`group_by_length` setting meant to fix that was silently ignored by the installed transformers.
This notebook does the grouping itself (a sampler that batches similar lengths), so it does not
depend on any library setting, and it **stops with an error at step 100 if the speed is still
too slow to finish**, instead of burning 12 GPU hours. Expected log: ~7-9 s/step.

- **Kaggle (recommended):** upload the jsonl as a dataset, enable GPU (T4 x2 or P100), then
  *Save Version -> Save & Run All*. It runs in the background with the browser closed (12 h limit).
- **Colab:** Runtime -> Change runtime type -> T4 GPU, then Run all. Keep the tab open: free Colab
  disconnects idle sessions. Checkpoints go to Google Drive, so if it disconnects just **Run all again**
  and training resumes from the last checkpoint.

Afterwards, score it locally with `scripts/eval_operator_agent.py` (held-out set, verified in Blender)
and compare with the 0.8B dry run's 86.2% and the earlier 2B runs: 74.0% (v1 data), 78.6% (v3 data), 91.3% (v4 data). Also score it on the broad set: `scripts/eval_broad.py --constrained` (v4: 53.5%).

## 1. Install

In [ ]:
%%capture
!pip install --upgrade unsloth

## 2. Settings, storage and data

On Colab this mounts Google Drive: checkpoints and the final GGUF are saved in
`MyDrive/blender_ai/operator_2b/`, so nothing is lost if the session dies.
Put `way1_train.jsonl` in that folder (or upload it when asked).

In [ ]:
import json, os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle T4 x2: Unsloth trains on one GPU; a second visible one can confuse it

MODEL_NAME = "unsloth/Qwen3.5-2B"
DATA_FILE = "way1_train_v5.jsonl"
EPOCHS = 2
VAL_SIZE = 350            # held back to watch for overfitting (same commands can appear in both splits;
                          # the real test is the held-out eval set, run locally afterwards)
GGUF_NAME = "qwen3.5-2b-operator-q8_0.gguf"

ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ  # not os.path.exists("/kaggle"): Colab has that folder too

if ON_KAGGLE:
    WORK = Path("/kaggle/working/operator_2b")
    found = list(Path("/kaggle/input").rglob(DATA_FILE))
    assert found, f"Add {DATA_FILE} as a Kaggle dataset first"
    DATASET_PATH = found[0]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/blender_ai/operator_2b")
    WORK.mkdir(parents=True, exist_ok=True)
    DATASET_PATH = WORK / DATA_FILE
    if not DATASET_PATH.exists():
        from google.colab import files
        print(f"Upload {DATA_FILE} (it is saved to Drive, so you only upload it once)")
        uploaded = files.upload()
        name = next(iter(uploaded))
        DATASET_PATH.write_bytes(uploaded[name])

WORK.mkdir(parents=True, exist_ok=True)
CHECKPOINTS = WORK / "checkpoints"

# Resume a run that was stopped: attach its checkpoint-NNNN folder as an input (Kaggle: upload it as a
# dataset), and training continues from there instead of starting over.
if ON_KAGGLE and not list(CHECKPOINTS.glob("checkpoint-*")):
    import shutil
    attached = sorted(Path("/kaggle/input").rglob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
    attached = [p for p in attached if p.is_dir() and (p / "trainer_state.json").exists()]
    if attached:
        CHECKPOINTS.mkdir(parents=True, exist_ok=True)
        shutil.copytree(attached[-1], CHECKPOINTS / attached[-1].name)
        print("Will resume from attached", attached[-1])
records = [json.loads(line) for line in DATASET_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
print(len(records), "examples from", DATASET_PATH)
print(records[0]["messages"][1]["content"][:300])
print("->", records[0]["messages"][2]["content"])

## 3. Load the model and attach LoRA

In [ ]:
from unsloth import FastModel
import torch

MAX_SEQ_LENGTH = 1536  # v5 big scenes reach ~3,100 characters (~1,100 tokens); answers must never be cut off

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,     # 2B fits in 16-bit on a T4 (~4 GB); avoids quantization noise
    full_finetuning=False,
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=32,                   # more capacity than the dry run's 16: 243 operators instead of ~20
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

## 4. Format with the chat template

Thinking is disabled: the operator agent answers with JSON immediately, and the same
`enable_thinking=False` is used at inference (`scripts/start_model_server.ps1` sets it).

In [ ]:
from datasets import Dataset

text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)  # multimodal processor -> its text tokenizer

def to_text(record):
    return {"text": text_tokenizer.apply_chat_template(
        record["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=False)}

dataset = Dataset.from_list([to_text(r) for r in records])
split = dataset.train_test_split(test_size=VAL_SIZE, seed=3407)
train_dataset, eval_dataset = split["train"], split["test"]
lengths = [len(text_tokenizer(t)["input_ids"]) for t in train_dataset.select(range(500))["text"]]
print(f"{len(train_dataset)} train / {len(eval_dataset)} val; tokens per example (sample): "
      f"median {sorted(lengths)[250]}, max {max(lengths)}")

## 5. Train (loss on the assistant reply only)

Saves a checkpoint every 200 steps. If a checkpoint already exists (the session died
and you pressed Run all again), training resumes from it instead of starting over.

In [ ]:
import random, time
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from unsloth.chat_templates import train_on_responses_only

TIME_BUDGET_H = 10.5  # Kaggle sessions end at ~12 h: stop training in time to save and export the model
MAX_PROJECTED_H = 9   # at step 100: if the run would take longer than this, stop with an error (something is wrong)

# Batch similar lengths together: v5 runs ~280-1,140 tokens, and random batches pad everything to the
# longest in the batch, ~2x the real work (21 s/step, ~14 h). Done here with our own sampler instead of
# transformers' group_by_length, whose name changes between versions and was silently ignored once.
class LengthGroupedSampler(torch.utils.data.Sampler):
    """Sort by length, cut into batches, shuffle the batch order (new order each epoch, same order on resume)."""
    def __init__(self, lengths, batch_size, seed):
        self.lengths, self.batch_size, self.seed, self.epoch = lengths, batch_size, seed, 0
    def set_epoch(self, epoch):
        self.epoch = int(epoch)
    def __len__(self):
        return len(self.lengths)
    def __iter__(self):
        order = sorted(range(len(self.lengths)), key=self.lengths.__getitem__)
        batches = [order[i:i + self.batch_size] for i in range(0, len(order), self.batch_size)]
        last = batches.pop() if len(batches[-1]) < self.batch_size else None  # a short batch must stay last,
        random.Random(self.seed + self.epoch).shuffle(batches)               # or every later batch shifts
        if last:
            batches.append(last)
        return iter([i for b in batches for i in b])


class GroupedSFTTrainer(SFTTrainer):
    def _get_train_sampler(self, *args, **kwargs):  # signature differs between transformers versions
        lengths = [len(ids) for ids in self.train_dataset["input_ids"]]
        print(f"Length-grouped batches: {len(lengths)} examples, {sum(lengths):,} tokens")
        return LengthGroupedSampler(lengths, self.args.per_device_train_batch_size, self.args.seed)


# Validation in length order too (evaluation never shuffles), so it isn't padded either.
eval_dataset = eval_dataset.map(lambda r: {"n": len(r["text"])}).sort("n").remove_columns("n")


class TimeBudget(TrainerCallback):
    """Stop cleanly before the session limit, so the model is still saved and exported."""
    def on_train_begin(self, args, state, control, **kwargs):
        self.start, self.first = time.time(), state.global_step  # first > 0 when resuming
    def on_step_end(self, args, state, control, **kwargs):
        hours = (time.time() - self.start) / 3600
        done = state.global_step - self.first
        if done in (20, 100):
            rate = (time.time() - self.start) / done
            left = (state.max_steps - state.global_step) * rate / 3600
            print(f"step {state.global_step}: {rate:.1f} s/step, ~{left:.1f} h to go")
            if done == 100 and hours + left > MAX_PROJECTED_H:
                raise RuntimeError(f"{rate:.1f} s/step means ~{hours + left:.0f} h (expected ~7-9 s/step, ~6 h). "
                                   "Batches are probably padded again: check the 'Length-grouped batches' line above.")
        if hours > TIME_BUDGET_H:
            print(f"Time budget reached at step {state.global_step}/{state.max_steps}: stopping and saving.")
            control.should_training_stop = True


trainer = GroupedSFTTrainer(
    model=model,
    tokenizer=text_tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[TimeBudget()],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=8,   # long big-scene examples: 8 x 2 keeps the effective batch at 16 within T4 memory
        gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        warmup_steps=50,
        lr_scheduler_type="cosine",
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=250,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=3407,
        output_dir=str(CHECKPOINTS),
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

existing = sorted(CHECKPOINTS.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1])) if CHECKPOINTS.exists() else []
if existing:
    print("Resuming from", existing[-1])
stats = trainer.train(resume_from_checkpoint=str(existing[-1]) if existing else None)
print(stats)
model.save_pretrained(str(WORK / "lora_adapter"))  # small backup of the trained adapter
text_tokenizer.save_pretrained(str(WORK / "lora_adapter"))

## 6. Quick check on held-back examples

Not the real score (those commands' other phrasings were in training), but it catches a
broken run: format errors, or answers that don't match.

In [ ]:
import re

FastModel.for_inference(model)
OP_RE = re.compile(r"^[a-z_][a-z0-9_]*\.[a-z_][a-z0-9_]*$")

def generate(messages):
    prompt = text_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = text_tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    return text_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

val_ids = set(split["test"]["text"])
val_records = [r for r, d in zip(records, dataset) if d["text"] in val_ids][:60]
valid = exact = 0
for record in val_records:
    reply = generate(record["messages"][:2])
    try:
        calls = json.loads(reply)["calls"]
        ok = all(OP_RE.match(c["op"]) for c in calls)
    except Exception:
        ok = False
    valid += ok
    exact += ok and json.loads(reply) == json.loads(record["messages"][2]["content"])
print(f"valid op/1: {valid}/{len(val_records)}, exact match: {exact}/{len(val_records)}")
for record in val_records[:8]:
    print(record["request"], "->", generate(record["messages"][:2]))

## 7. Export to GGUF

Merges the LoRA into the base model and writes `q8_0` (~2 GB). On Colab it is copied to
your Drive folder (download it from drive.google.com: more reliable than a 2 GB browser
download). On Kaggle it appears in the Output tab.

In [ ]:
import glob, shutil

model.save_pretrained_gguf("operator_gguf", tokenizer, quantization_method="q8_0")
candidates = [f for f in glob.glob("**/*.gguf", recursive=True) if "mmproj" not in f.lower()]
assert candidates, "No GGUF found -- check the export output above for errors"
gguf = max(candidates, key=os.path.getsize)  # the language model, not a vision projector
target = WORK / GGUF_NAME
shutil.copy(gguf, target)
print(f"Saved {target} ({target.stat().st_size / 1e9:.2f} GB)")
if ON_KAGGLE:  # keep the Output tab to the files you need
    for folder in ("operator_gguf", "operator_gguf_gguf", str(CHECKPOINTS)):  # merged weights, duplicate GGUF + mmproj, checkpoints
        shutil.rmtree(folder, ignore_errors=True)
    print("Download it from the Output tab of this notebook version:", target)

## 8. On your machine

```powershell
# put the GGUF in models/operator/, then:
.\scripts\start_model_server.ps1 -Model models/operator/qwen3.5-2b-operator-q8_0.gguf

# held-out score (runs each answer in headless Blender); compare with the 0.8B dry run's 86.2%:
python scripts/eval_operator_agent.py --base-url http://127.0.0.1:8080/v1
```